# 第9章: 事前学習済み言語モデル（BERT型）

本章では、BERT型の事前学習済みモデルを利用して、マスク単語の予測や文ベクトルの計算、評判分析器（ポジネガ分類器）の構築に取り組む。

## 80. トークン化

"The movie was full of incomprehensibilities."という文をトークンに分解し、トークン列を表示せよ。

In [ ]:
!pip install tiktoken

In [ ]:
import tiktoken

text_to_tokenize = "The movie was full of incomprehensibilities."

encoding = tiktoken.get_encoding("cl100k_base")
tokens = encoding.encode(text_to_tokenize)

# 各トークンIDを文字列にデコードして表示
token_strings = [encoding.decode([t]) for t in tokens]

print(f"トークン列: {tokens}")
print(f"トークン文字列: {token_strings}")
print(f"トークン数: {len(tokens)}")

トークン列: [791, 5818, 574, 2539, 315, 53990, 31882, 729, 13757, 13]
トークン文字列: ['The', ' movie', ' was', ' full', ' of', ' incom', 'preh', 'ens', 'ibilities', '.']
トークン数: 10


## 81. マスクの予測

"The movie was full of [MASK]."の"[MASK]"を埋めるのに最も適切なトークンを求めよ。

In [ ]:
!pip install transformers torch

In [ ]:
from transformers import pipeline

# マスク予測用のパイプラインを初期化 (英語の基本モデルを使用)
mask_filler = pipeline("fill-mask", model="bert-base-uncased")

# 予測したい文
text = "The movie was full of [MASK]."

# 予測の実行
results = mask_filler(text)

# 結果の表示
print(f"入力文: {text}\n")
for result in results:
    print(f"スコア: {result['score']:.4f}, トークン: {result['token_str']}, 文: {result['sequence']}")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

入力文: The movie was full of [MASK].

スコア: 0.1071, トークン: fun, 文: the movie was full of fun.
スコア: 0.0663, トークン: surprises, 文: the movie was full of surprises.
スコア: 0.0447, トークン: drama, 文: the movie was full of drama.
スコア: 0.0272, トークン: stars, 文: the movie was full of stars.
スコア: 0.0254, トークン: laughs, 文: the movie was full of laughs.


## 82. マスクのtop-k予測

"The movie was full of [MASK]."の"[MASK]"に埋めるのに適切なトークン上位10個と、その確率（尤度）を求めよ。

In [ ]:
from transformers import pipeline

# マスク予測用のパイプラインを初期化
mask_filler = pipeline("fill-mask", model="bert-base-uncased")

# 予測したい文
text = "The movie was full of [MASK]."

# 上位10個の予測を取得
results = mask_filler(text, top_k=10)

# 結果の表示
print(f"入力文: {text}\n")
for i, result in enumerate(results, 1):
    print(f"{i:2}: スコア {result['score']:.4f} | トークン: {result['token_str']}")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


入力文: The movie was full of [MASK].

 1: スコア 0.1071 | トークン: fun
 2: スコア 0.0663 | トークン: surprises
 3: スコア 0.0447 | トークン: drama
 4: スコア 0.0272 | トークン: stars
 5: スコア 0.0254 | トークン: laughs
 6: スコア 0.0195 | トークン: action
 7: スコア 0.0190 | トークン: excitement
 8: スコア 0.0183 | トークン: people
 9: スコア 0.0150 | トークン: tension
10: スコア 0.0146 | トークン: music


## 83. CLSトークンによる文ベクトル

以下の文の全ての組み合わせに対して、最終層の[CLS]トークンの埋め込みベクトルを用いてコサイン類似度を求めよ。

- "The movie was full of fun."
- "The movie was full of excitement."
- "The movie was full of crap."
- "The movie was full of rubbish."


### [CLS]トークンの抽出例

各文章の先頭にある `[CLS]` トークン（ID: 101）に対応する、最終層の隠れ状態（Hidden State）を取得します。

In [ ]:
import torch
from transformers import BertTokenizer, BertModel
from sklearn.metrics.pairwise import cosine_similarity

# モデルとトークナイザーの準備
model_name = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name)

sentences = [
    "The movie was full of fun.",
    "The movie was full of excitement.",
    "The movie was full of crap.",
    "The movie was full of rubbish."
]

# [CLS] トークンのベクトルを取得する関数
def get_cls_embedding(text):
    inputs = tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
    # outputs.last_hidden_state の形状は [バッチサイズ, 系列長, 隠れ層の次元数]
    # [CLS]は先頭（インデックス0）に存在します
    cls_vec = outputs.last_hidden_state[0, 0, :]
    return cls_vec

# 各文のベクトルを計算
embeddings = [get_cls_embedding(s) for s in sentences]

# 類似度の計算と表示
print("--- コサイン類似度 ([CLS]トークン) ---")
for i in range(len(sentences)):
    for j in range(i + 1, len(sentences)):
        sim = cosine_similarity(embeddings[i].reshape(1, -1), embeddings[j].reshape(1, -1))[0][0]
        print(f"{sentences[i]} vs {sentences[j]}: {sim:.4f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


--- コサイン類似度 ([CLS]トークン) ---
The movie was full of fun. vs The movie was full of excitement.: 0.9881
The movie was full of fun. vs The movie was full of crap.: 0.9558
The movie was full of fun. vs The movie was full of rubbish.: 0.9475
The movie was full of excitement. vs The movie was full of crap.: 0.9541
The movie was full of excitement. vs The movie was full of rubbish.: 0.9487
The movie was full of crap. vs The movie was full of rubbish.: 0.9807


## 84. 平均による文ベクトル

以下の文の全ての組み合わせに対して、最終層の埋め込みベクトルの平均を用いてコサイン類似度を求めよ。

- "The movie was full of fun."
- "The movie was full of excitement."
- "The movie was full of crap."
- "The movie was full of rubbish."

In [2]:
import torch
from transformers import BertTokenizer, BertModel
from sklearn.metrics.pairwise import cosine_similarity

# モデルとトークナイザーの準備
model_name = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name)

sentences = [
    "The movie was full of fun.",
    "The movie was full of excitement.",
    "The movie was full of crap.",
    "The movie was full of rubbish."
]

# 全てのトークンの埋め込みベクトルの平均を取得する関数
def get_mean_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
    # 最終層の隠れ状態の平均を計算
    # outputs.last_hidden_state: [バッチサイズ, 系列長, 隠れ層の次元数]
    mean_vec = outputs.last_hidden_state[0].mean(dim=0)
    return mean_vec

# 各文のベクトルを計算
embeddings_mean = [get_mean_embedding(s) for s in sentences]

# 類似度の計算と表示
print("--- コサイン類似度 (全トークンの平均) ---")
for i in range(len(sentences)):
    for j in range(i + 1, len(sentences)):
        sim = cosine_similarity(embeddings_mean[i].reshape(1, -1), embeddings_mean[j].reshape(1, -1))[0][0]
        print(f"{sentences[i]} vs {sentences[j]}: {sim:.4f}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


--- コサイン類似度 (全トークンの平均) ---
The movie was full of fun. vs The movie was full of excitement.: 0.9568
The movie was full of fun. vs The movie was full of crap.: 0.8490
The movie was full of fun. vs The movie was full of rubbish.: 0.8169
The movie was full of excitement. vs The movie was full of crap.: 0.8352
The movie was full of excitement. vs The movie was full of rubbish.: 0.7938
The movie was full of crap. vs The movie was full of rubbish.: 0.9226


## 85. データセットの準備

[General Language Understanding Evaluation (GLUE)](https://gluebenchmark.com/) ベンチマークで配布されている[Stanford Sentiment Treebank (SST)](https://dl.fbaipublicfiles.com/glue/data/SST-2.zip) から訓練セット（train.tsv）と開発セット（dev.tsv）のテキストと極性ラベルと読み込み、さらに全てのテキストはトークン列に変換せよ。

In [3]:
!pip install datasets

In [5]:
from datasets import load_dataset
from transformers import BertTokenizer

# 1. SST-2 データセットの読み込み
# リポジトリ名を明示的に指定してエラーを回避します
dataset = load_dataset("nyu-mll/glue", "sst2")

# 2. トークナイザーの準備
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# 3. トークン化関数の定義
def tokenize_function(examples):
    # truncation=True で最大長を超える場合に切り捨てを行います
    return tokenizer(examples["sentence"], truncation=True)

# 4. データセット全体にトークン化を適用
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# データの確認
print("--- 訓練データの事例 (トークン化後) ---")
print(tokenized_datasets["train"][0])

# トークンIDを文字列に戻して確認
print("\n--- デコード結果 ---")
print(tokenizer.decode(tokenized_datasets["train"][0]["input_ids"]))

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

sst2/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

...

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

--- 訓練データの事例 (トークン化後) ---
{'sentence': 'hide new secretions from the parental units ', 'label': 0, 'idx': 0, 'input_ids': [101, 5342, 2047, 3595, 8496, 2013, 1996, 18643, 3197, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

--- デコード結果 ---
[CLS] hide new secretions from the parental units [SEP]


## 86. ミニバッチの作成

85で読み込んだ訓練データの一部（例えば冒頭の4事例）に対して、パディングなどの処理を行い、トークン列の長さを揃えてミニバッチを構成せよ。

In [6]:
from transformers import DataCollatorWithPadding
import torch

# 1. データコレーター（パディング用）の準備
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 2. 訓練データの冒頭4事例を抽出
# dataset.mapで作成した tokenized_datasets を使用します
samples = tokenized_datasets["train"].select(range(4))

# 余計な列（textなど）を除外して、モデルが必要な列だけに絞る
# 'input_ids', 'token_type_ids', 'attention_mask' などが含まれます
features = [{k: v for k, v in sample.items() if k in ['input_ids', 'token_type_ids', 'attention_mask', 'label']} for sample in samples]

# 3. ミニバッチの作成（パディングの適用）
batch = data_collator(features)

# 4. 結果の確認
print(f"ミニバッチのキー: {batch.keys()}")
print(f"input_ids の形状: {batch['input_ids'].shape}")
print("\n--- 各データの元の長さ ---")
for i, s in enumerate(samples):
    print(f"事例 {i}: {len(s['input_ids'])} トークン")

print("\n--- パディング後の input_ids ---")
print(batch['input_ids'])

# デコードして [PAD] が追加されているか確認
print("\n--- デコード結果 (パディング確認) ---")
print(tokenizer.decode(batch['input_ids'][0]))

ミニバッチのキー: KeysView({'input_ids': tensor([[  101,  5342,  2047,  3595,  8496,  2013,  1996, 18643,  3197,   102,
             0,     0,     0,     0,     0],
        [  101,  3397,  2053, 15966,  1010,  2069,  4450,  2098, 18201,  2015,
           102,     0,     0,     0,     0],
        [  101,  2008,  7459,  2049,  3494,  1998, 10639,  2015,  2242,  2738,
          3376,  2055,  2529,  3267,   102],
        [  101,  3464, 12580,  8510,  2000,  3961,  1996,  2168,  2802,   102,
             0,     0,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0

## 87. ファインチューニング

訓練セットを用い、事前学習済みモデルを極性分析タスク向けにファインチューニングせよ。検証セット上でファインチューニングされたモデルの正解率を計測せよ。

In [9]:
import torch
from transformers import BertForSequenceClassification, BertTokenizer, TrainingArguments, Trainer, DataCollatorWithPadding
from datasets import load_dataset
import evaluate
import numpy as np

# 1. データの再ロードとトークナイズ (未定義エラー防止)
dataset = load_dataset("nyu-mll/glue", "sst2")
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_function(examples):
    return tokenizer(examples["sentence"], truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 2. 分類用モデルの準備
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

# 3. 評価指標の設定
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# 4. 学習パラメータの設定
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=100,
    report_to="none"
)

# 5. Trainerの初期化 (引数の構成を確認)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer, # tokenizer引数の代わりにprocessing_classを使用するか、DataCollatorに任せる
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# 6. 学習の実行
trainer.train()

# 7. 評価
eval_results = trainer.evaluate()
print(f"\n検証セットでの正解率 (Accuracy): {eval_results['eval_accuracy']:.4f}")

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.153606,0.242435,0.924312


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


検証セットでの正解率 (Accuracy): 0.9243


## 88. 極性分析

問題87でファインチューニングされたモデルを用いて、以下の文の極性を予測せよ。

- "The movie was full of incomprehensibilities."
- "The movie was full of fun."
- "The movie was full of excitement."
- "The movie was full of crap."
- "The movie was full of rubbish."


In [12]:
import torch
import torch.nn.functional as F
from transformers import BertTokenizer

# デバイスの判定
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# トークナイザーの再準備
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# 予測対象の文
test_sentences = [
    "The movie was full of incomprehensibilities.",
    "The movie was full of fun.",
    "The movie was full of excitement.",
    "The movie was full of crap.",
    "The movie was full of rubbish."
]

# モデルを適切なデバイスに移動し、評価モードに設定
model.to(device)
model.eval()

print(f"{'Sentence':<50} | {'Prediction':<10} | {'Confidence':<10}")
print("-" * 75)

for text in test_sentences:
    # トークナイズし、入力をモデルと同じデバイスに移動
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)

    # 予測（勾配計算を無効化）
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = F.softmax(logits, dim=-1)
        prediction = torch.argmax(probs, dim=-1).item()
        confidence = probs[0][prediction].item()

    # SST-2のラベル: 0 -> Negative, 1 -> Positive
    label_map = {0: "Negative", 1: "Positive"}

    print(f"{text:<50} | {label_map[prediction]:<10} | {confidence:.4f}")

Sentence                                           | Prediction | Confidence
---------------------------------------------------------------------------
The movie was full of incomprehensibilities.       | Negative   | 0.9970
The movie was full of fun.                         | Positive   | 0.9989
The movie was full of excitement.                  | Positive   | 0.9982
The movie was full of crap.                        | Negative   | 0.9978
The movie was full of rubbish.                     | Negative   | 0.9980


## 89. アーキテクチャの変更

問題87とは異なるアーキテクチャ（例えば[CLS]トークンを用いるか、各トークンの最大値プーリングを用いるなど）の分類モデルを設計し、事前学習済みモデルを極性分析タスク向けにファインチューニングせよ。検証セット上でファインチューニングされたモデルの正解率を計測せよ。

In [14]:
import torch
import torch.nn as nn
from transformers import BertModel, BertTokenizer, TrainingArguments, Trainer, DataCollatorWithPadding
from datasets import load_dataset
import evaluate
import numpy as np

# 1. データセットの再準備
dataset = load_dataset("nyu-mll/glue", "sst2")
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_function(examples):
    return tokenizer(examples["sentence"], truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# 2. カスタムモデルの定義 (Max Poolingを使用)
class BertForSequenceClassificationWithMaxPooling(nn.Module):
    def __init__(self, model_name='bert-base-uncased', num_labels=2):
        super().__init__()
        self.bert = BertModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)
        self.num_labels = num_labels

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None, labels=None, **kwargs):
        outputs = self.bert(input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        last_hidden_state = outputs.last_hidden_state

        expanded_mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        masked_hidden_state = last_hidden_state.masked_fill(expanded_mask == 0, -1e9)

        pooled_output = torch.max(masked_hidden_state, dim=1)[0]
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)

        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))

        return {"loss": loss, "logits": logits} if loss is not None else {"logits": logits}

custom_model = BertForSequenceClassificationWithMaxPooling()

# 3. 学習の設定
training_args_89 = TrainingArguments(
    output_dir="./results_maxpooling",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=100,
    report_to="none"
)

# 4. Trainerの実行 (tokenizer引数をprocessing_classに修正)
trainer_89 = Trainer(
    model=custom_model,
    args=training_args_89,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer_89.train()
eval_results_89 = trainer_89.evaluate()
print(f"\nMax Poolingモデルの検証セット正解率: {eval_results_89['eval_accuracy']:.4f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.146633,0.247188,0.924312



Max Poolingモデルの検証セット正解率: 0.9243
